# Unity Environment Walkthrough

This notebook demonstrates how to connect the TensorAeroSpace Unity environment (Editor or standalone build) to Python via `mlagents`.

**Unity Environment Repository:** [TensorAeroSpace/UnityAirplaneEnvironment](https://github.com/TensorAeroSpace/UnityAirplaneEnvironment)

## Overview

1. Install dependencies (preferably in a fresh `venv`/`conda`)
2. Specify the path to the Unity build or leave `None` to connect to the Editor
3. Create `UnityEnvironment`, configure visualization/server mode flags, and wrap it in `gym`
4. (Optional) Apply action discretization via `unity_discrete_env`
5. Execute a few steps and properly close the connection

> ℹ️ If running in headless/server mode, set `SERVER_MODE=True` (see below) and ensure the Unity build was created with the "Run In Background" flag.

## Related Examples

- [SAC with Unity](../example-sac-unity.ipynb) — continuous control with SAC agent
- [DQN with Unity](../reinforcement_learning/example_dqn_unity.ipynb) — discrete control with DQN agent


In [ ]:
!pip install mlagents==1.1.0


## 1. Installing Dependencies

Run this cell **after** activating a virtual environment (`python -m venv .venv && source .venv/bin/activate` or `conda create -n tas python=3.10 && conda activate tas`).

- `mlagents` and `mlagents_envs` must match the version used to build the Unity project (the repository uses 1.1.0).
- To work with TensorAeroSpace, also install `gym==0.20.0` and `gym-unity==0.28.0` (see `docs/ru/guide/unity_env.md`).
- When running on a server without a display, additionally install `xvfb` or use a headless build.

In [ ]:
from pathlib import Path
from typing import Optional

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper


def get_plane_env(
    env_path: Optional[str] = "",
    server: bool = False,
    worker: int = 0,
    log_dir: str = "/app/logs",
    additional_args: Optional[list[str]] = None,
):
    """Создаёт gym-обёртку для UnityAirplaneEnvironment.

    Args:
        env_path: путь к собранному билду. `None`/"" → подключение к Editor.
        server: `True` для headless-запуска (отключает графику в Unity).
        worker: уникальный ID для параллельных копий (избегает конфликтов портов).
        log_dir: каталог, куда Unity будет писать логи.
        additional_args: дополнительные аргументы для процесса Unity (например, `-logfile`).
    """

    resolved_path = "" if env_path in (None, "") else str(env_path)
    log_dir_path = Path(log_dir)
    log_dir_path.mkdir(parents=True, exist_ok=True)
    unity_args = additional_args or ["-logfile", str(log_dir_path / "unity.log")]

    print(
        f"Connecting to {'Unity Editor' if resolved_path == '' else resolved_path} | "
        f"server={server} | worker_id={worker} | logs={log_dir_path}"
    )

    unity_env = UnityEnvironment(
        resolved_path,
        worker_id=worker,
        no_graphics=server,
        log_folder=str(log_dir_path),
        additional_args=unity_args,
    )

    env = UnityToGymWrapper(unity_env, uint8_visual=True)
    return env


In [ ]:
# ↳ Укажите путь к билду или оставьте None для подключения к редактору
ENV_PATH = None  # Например: Path("/tf/linux_build/build.x86_64")

# Включайте SERVER_MODE=True, если запускаете на сервере без окна
SERVER_MODE = False

# Используйте разные worker_id при параллельных экспериментах
WORKER_ID = 0

# Папка под логи Unity (создаётся автоматически)
LOG_DIR = Path("/app/logs")
ADDITIONAL_ARGS = ["-logfile", str(LOG_DIR / "unity.log")]


## 2. Connection Configuration

Fill in the variables below for your scenario:

- `ENV_PATH` -- absolute path to the built Unity build (`*.x86_64`, `.app`, `.exe`). If you want to connect to the Unity Editor, leave `None` and open the scene in Play mode.
- `SERVER_MODE` -- `True` if running without a window/on a server (enables `no_graphics=True`).
- `WORKER_ID` -- integer for parallel runs (different IDs prevent port conflicts).
- `LOG_DIR` -- directory where Unity will write logs (default `/app/logs/`).

On first run it is useful to verify that:
1. The Unity scene has `Behavior Parameters -> Behavior Name` matching what Python expects.
2. The scene is added to the list in Build Settings.
3. ML-Agents version 2.2.1-exp.1 (as in the example) is compatible with the installed Python packages.

## 3. Creating `UnityEnvironment` and the `gym` Wrapper

The `get_plane_env` call below:

1. Creates the log directory (if needed) and passes `-logfile` to Unity.
2. Automatically switches between Unity Editor (when `ENV_PATH=None`) and a standalone build.
3. Returns a `UnityToGymWrapper` compatible with TensorAeroSpace agents.

If the connection is not established:
- Make sure the Editor is in Play mode (when `ENV_PATH=None`).
- Check that the port is free or change `WORKER_ID`.
- On Linux, grant executable permissions to the build file (`chmod +x`).

In [ ]:
env = get_plane_env(
    env_path=ENV_PATH,
    server=SERVER_MODE,
    worker=WORKER_ID,
    log_dir=str(LOG_DIR),
    additional_args=ADDITIONAL_ARGS,
)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)


## 4. Action Discretization (optional)

`unity_discrete_env` converts the multi-dimensional action (7 channels with 3 values each) into a single integer `0 ... 3^7-1`. This is convenient for algorithms expecting a discrete action space (DQN, A3C, etc.).

If you need the continuous version, skip this step and work directly with `env`.

In [ ]:
from tensoraerospace.envs.unity_env import unity_discrete_env

# Преобразуем непрерывное действие (7 каналов) в один дискретный индекс
wrapped_env = unity_discrete_env(env)
print("Discrete action space:", wrapped_env.action_space)


## 5. Connection Smoke-test

Below we verify the basic `reset -> step -> close` cycle:

1. `reset()` should return observations without errors. If you get a `UnityEnvironmentException`, check the log (`LOG_DIR`).
2. `step(action)` uses a discrete action `1` (aileron channel). Replace with `wrapped_env.action_space.sample()` or an agent action.
3. After testing, always call `close()`, otherwise the Unity process will remain running and block the port.

> ❗ If Unity reports "Display 1 No cameras rendering", open the scene and make sure the active camera is bound to Display 1 (see the checklist in `docs/ru/guide/unity_env.md`).

In [ ]:
initial_obs = wrapped_env.reset()
print("Initial observation shape:", getattr(initial_obs, "shape", type(initial_obs)))

sample_action = wrapped_env.action_space.sample()
print("Sample action:", sample_action)

observation, reward, done, info = wrapped_env.step(sample_action)
print(
    f"Reward={reward:.3f} | done={done} | info keys={list(info.keys()) if isinstance(info, dict) else info}"
)

wrapped_env.close()
print("Unity environment closed cleanly ✅")
